# Evaporating Universe — Paper I
## NB05_MCMC_D2: Chain Analysis + Cross-Check — Run D1

**Purpose:** Reproducibility & Transparency for Referees

| Item | Value |
|:-----|:------|
| Run | D1 — EU params FIXED, full IDE, +SH0ES |
| EU params | FIXED from NB02 predictions (not sampled) |
| IDE perturbations | `eu_has_ide_perturbations` = value from YAML (verified in §3) |
| Datasets | Planck NPIPE CamSpec TTTEEE + lowl TT/EE + lensing + DESI DR2 + Pantheon+ |
| Chains | 8 MPI chains, R-1 target < 0.01 |
| Infrastructure | AWS m7a.48xlarge (192 vCPU) |
| Theory code | CLASS v3.3.4 + EU patch vf1.3 |

**D1 vs D2:** D1 uses `shoes_lki.SH0ES_LKI` as an additional likelihood constraint. D2 extends further with DES-Y3 S₈ proxy. Both use full IDE (`eu_has_ide_perturbations=1`) with EU parameters fixed from NB02.

**Workflow:**
1. §1: Upload ALL files (patch + chains + YAML) — one dialog
2. §2: Environment setup (Cobaya + CLASS EU + likelihoods) — ~15 min
3. §3: Display YAML configuration (exact run conditions + perturbation flag)
4. §4: Convergence diagnostics (R-1 + ESS + trace plots)
5. §5: Corner plots (GetDist)
6. §6: Parameter table + best-fit
7. §7: Cross-check — evaluate χ² at best-fit point
8. §8: Export JSON (referee-grade)
9. §9: Download

**Referee instructions:** Upload files in §1, then Run All. §2 takes ~15 min. §7 takes ~2 min.

**No hardcoded theory values. No fallbacks. All parameters extracted from YAML.**

---


In [ ]:
# ============================================================
# §1. UPLOAD ALL FILES
# ============================================================
# Upload everything needed in ONE step:
#   - patch_class.py (EU modification for CLASS)
#   - 8 chain files (eu_NB05D2.1.txt ... eu_NB05D2.8.txt)
#   - eu_NB05D2.updated.yaml (run configuration)
#
# Tip: zip all 10 files into one archive for convenience.
# ============================================================
import os, zipfile

CHAIN_ROOT = 'eu_NB05D2'
N_CHAINS = 8

ALL_REQUIRED = (
    ['patch_class.py']
    + [f'{CHAIN_ROOT}.{i}.txt' for i in range(1, N_CHAINS + 1)]
    + [f'{CHAIN_ROOT}.updated.yaml']
    + [f'{CHAIN_ROOT}.checkpoint']
    + ['shoes_lki.py']
)

print('=' * 60)
print('REQUIRED FILES (upload all at once)')
print('  Include .checkpoint file for convergence verification')
print('=' * 60)
for f in ALL_REQUIRED:
    status = '\u2705' if os.path.exists(f'/content/{f}') else '\u274c'
    print(f'  {status} {f}')

missing = [f for f in ALL_REQUIRED if not os.path.exists(f'/content/{f}')]
if missing:
    print(f'\n  {len(missing)} files missing — opening upload dialog...')
    print(f'  Tip: zip all files and upload the zip.\n')
    from google.colab import files as _files
    uploaded = _files.upload()
    for name, content in uploaded.items():
        with open(f'/content/{name}', 'wb') as f:
            f.write(content)
        if name.endswith('.zip'):
            with zipfile.ZipFile(f'/content/{name}', 'r') as zf:
                zf.extractall('/content')
            print(f'  [OK] {name} extracted')
        else:
            print(f'  [OK] {name}')

# Final verification
print(f'\n  Verification:')
all_ok = True
for f in ALL_REQUIRED:
    exists = os.path.exists(f'/content/{f}')
    status = '\u2705' if exists else '\u274c MISSING'
    print(f'    {status} {f}')
    if not exists:
        all_ok = False

assert all_ok, 'FATAL: Not all files present. Re-run this cell.'
print(f'\n[OK] All {len(ALL_REQUIRED)} files present')


---


In [ ]:
# ============================================================
# §2. ENVIRONMENT SETUP
# ============================================================
# Installs: Cobaya, GetDist, CLASS EU (patched), Planck+DESI+SNe
# Time: ~15 min on Colab (compile + download)
# ============================================================
import os, subprocess, sys, time
import numpy as np

t0_setup = time.time()

PACKAGES_PATH = '/content/packages'
CLASS_DIR = '/content/class_eu'
PATCH_SCRIPT = '/content/patch_class.py'

# -- Step 1: Python packages --
print('=' * 60)
print('[1/5] Installing Python packages...')
os.system('pip install -q cobaya getdist pyyaml matplotlib numpy scipy cython')

import cobaya
print(f'  [OK] Cobaya {cobaya.__version__}')
import getdist
print(f'  [OK] GetDist {getdist.__version__}')

# -- Step 2: Clone CLASS v3.3.4 --
print('\n[2/5] Cloning CLASS v3.3.4...')
if not os.path.isdir(CLASS_DIR):
    os.system(f'git clone --branch v3.3.4 --depth 1 https://github.com/lesgourg/class_public.git {CLASS_DIR} 2>&1 | tail -2')
    print('  [OK] CLASS cloned')
else:
    print('  [SKIP] CLASS already present')

# -- Step 3: Apply EU patch --
print('\n[3/5] Applying EU patches to CLASS...')
assert os.path.exists(PATCH_SCRIPT), f'FATAL: {PATCH_SCRIPT} not found. Re-run §1.'
os.system(f'cd /content && python3 {PATCH_SCRIPT} {CLASS_DIR}')

# -- Step 4: Compile CLASS + install classy --
print('\n[4/5] Compiling CLASS-EU...')
os.system(f'cd {CLASS_DIR} && make clean > /dev/null 2>&1; make -j$(nproc) 2>&1 | tail -3')
assert os.path.exists(f'{CLASS_DIR}/libclass.a'), 'FATAL: libclass.a not built!'
os.system(f'cd {CLASS_DIR} && pip install -q --no-build-isolation . 2>&1 | tail -2')

# Smoke test
result = subprocess.run(
    [sys.executable, '-c', """
from classy import Class
c = Class()
c.set({"output":"tCl", "l_max_scalars":100,
       "100*theta_s": 1.0411,
       "omega_b": 0.02237, "omega_cdm": 0.12,
       "eu_epsilon_ir": 0.04264, "eu_z_trans": 5.986, "eu_b": 0.5278,
       "Omega_Lambda": 0, "w0_fld": -1.0, "wa_fld": 0, "cs2_fld": 1.0,
       "eu_has_ide_perturbations": 0,
       "N_ur": 2.0328, "N_ncdm": 1, "m_ncdm": 0.0589})
c.compute()
print(f"H0={c.h()*100:.2f}")
c.struct_cleanup(); c.empty()
"""],
    capture_output=True, text=True
)
assert 'H0=' in result.stdout, f'FATAL: CLASS smoke test failed!\n{result.stderr}'
print(f'  [OK] CLASS-EU smoke test: {result.stdout.strip()}')

# -- Step 5: Install likelihood data --
print('\n[5/5] Installing likelihood data (Planck NPIPE + DESI DR2 + Pantheon+)...')
os.system(f"""cobaya-install \\
    planck_NPIPE_highl_CamSpec.TTTEEE \\
    planck_2018_lowl.TT \\
    planck_2018_lowl.EE \\
    planck_2018_lensing.clik \\
    sn.pantheonplus \\
    bao.desi_dr2 \\
    -p {PACKAGES_PATH} 2>&1 | tail -5""")

# Install clipy for Planck lensing
os.system(f'cobaya-install planck_2018_lensing.clik -p {PACKAGES_PATH} 2>&1 | tail -3')

dt_setup = time.time() - t0_setup
print(f'\n{"="*60}')
print(f'[OK] Setup complete in {dt_setup:.0f}s')
print(f'  CLASS-EU: {CLASS_DIR}')
print(f'  Packages: {PACKAGES_PATH}')
print(f'  Cobaya: {cobaya.__version__}')
print(f'={"="*59}')


---


In [ ]:
# ============================================================
# §2.5. COCOA SETUP — Build DES-Y3 + CLASS-EU inside conda
# ============================================================
# Self-contained: writes build script to disk, then executes.
# Builds: Miniforge + conda env + Cocoa/CosmoLike + CLASS-EU
# Time: ~20-30 min on Colab
# ============================================================

import os, subprocess, time, shutil, glob

t0_cocoa = time.time()

print('=' * 60)
print('§2.5 — COCOA + CLASS-EU SETUP (conda environment)')
print('=' * 60)

CONDA_DIR = '/content/miniforge3'
COCOA_INNER = '/content/cocoa/Cocoa'
DES_Y3_DIR = f'{COCOA_INNER}/projects/des_y3'
DES_Y3_DATA = f'{DES_Y3_DIR}/data'
CLASS_CONDA = '/content/class_eu_conda'

# ── Write build script to disk ──
print('\n  Writing setup_cocoa_conda.sh...')
_setup_script = '''\
#!/bin/bash
# ============================================================
# setup_cocoa_conda.sh — Build Cocoa + CLASS-EU inside conda
# ============================================================
# Called from NB05_MCMC_D2.ipynb §2.5 via subprocess.
# Installs everything needed for DES-Y3 cross-check in conda.
#
# Usage: bash setup_cocoa_conda.sh
# Env vars required:
#   CLASS_EU_PATCH=/content/patch_class.py
#   SHOES_LKI_SRC=/content/shoes_lki.py
# ============================================================
set -eo pipefail

CONDA_DIR="/content/miniforge3"
COCOA_DIR="/content/cocoa"
COCOA_INNER="$COCOA_DIR/Cocoa"
DES_Y3_DIR="$COCOA_INNER/projects/des_y3"
CLASS_CONDA="/content/class_eu_conda"
PACKAGES_PATH="/content/packages"

log() { echo "[cocoa-setup] [$(date '+%H:%M:%S')] $1"; }

# ── 1. Miniforge ──
log "Step 1/7: Miniforge"
if [ -d "$CONDA_DIR" ]; then
    log "  Already installed"
else
    wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    bash /tmp/miniforge.sh -b -u -p "$CONDA_DIR" > /dev/null 2>&1
    rm /tmp/miniforge.sh
    log "  Installed"
fi
export PATH="$CONDA_DIR/bin:$PATH"
eval "$($CONDA_DIR/bin/conda shell.bash hook)"

# ── 2. Conda env ──
log "Step 2/7: Conda env 'cocoa'"
if conda env list 2>/dev/null | grep -q "cocoa"; then
    log "  Already exists"
else
    wget -q https://raw.githubusercontent.com/CosmoLike/cocoa/refs/heads/main/cocoapy310.yml -O /tmp/cocoapy310.yml
    conda env create --name cocoa --file /tmp/cocoapy310.yml -q 2>&1 | tail -3
    rm /tmp/cocoapy310.yml
    log "  Created"
fi
conda activate cocoa

# Install cobaya + deps inside conda (NOT in cocoapy310.yml)
pip install -q cobaya pyyaml getdist iminuit 2>&1 | tail -2
log "  cobaya installed in conda"

# ── 3. Clone Cocoa + DES-Y3 ──
log "Step 3/7: Clone Cocoa + DES-Y3"
if [ -d "$COCOA_INNER" ]; then
    log "  Cocoa present"
else
    git clone --depth 1 https://github.com/CosmoLike/cocoa.git "$COCOA_DIR" 2>&1 | tail -2
    log "  Cocoa cloned"
fi
if [ -d "$DES_Y3_DIR" ]; then
    log "  DES-Y3 present"
else
    cd "$COCOA_INNER/projects"
    git clone --depth 1 https://github.com/CosmoLike/cocoa_des_y3.git des_y3 2>&1 | tail -2
    log "  DES-Y3 cloned"
fi

# ── 4. Write minimal set_installation_options.sh ──
log "Step 4/7: Build config"
cat > "$COCOA_INNER/set_installation_options.sh" << 'SETOPT'
#!/bin/bash
source "$(pwd -P)/installation_scripts/flags_impl_unset_keys.sh"
export ROOTDIR=$(pwd -P)
export IGNORE_BICEP_CMB_DATA=1
export IGNORE_SIMONS_OBSERVATORY_CMB_DATA=1
export IGNORE_CAMSPEC_CMB_DATA=1
export IGNORE_LIPOP_CMB_DATA=1
export IGNORE_COSMOPOWER_DATA=1
export IGNORE_ACTDR6_DATA=1
export IGNORE_HOLICOW_STRONG_LENSING_DATA=1
export IGNORE_SPT_CMB_DATA=1
export IGNORE_CAMB_CODE=1
export IGNORE_CLASS_CODE=1
export IGNORE_POLYCHORD_SAMPLER_CODE=1
export IGNORE_PLANCK_LIKELIHOOD_CODE=1
export IGNORE_ACTDR4_CODE=1
export IGNORE_ACTDR6_CODE=1
export IGNORE_CPP_CUBA_INSTALLATION=1
export IGNORE_FGSPECTRA_CODE=1
export IGNORE_VELOCILEPTORS_CODE=1
export IGNORE_SIMONS_OBSERVATORY_LIKELIHOOD_CODE=1
export IGNORE_CAMSPEC_LIKELIHOOD_CODE=1
export IGNORE_COSMOPOWER_CODE=1
export IGNORE_COSMOREC_CODE=1
export IGNORE_HYREC2_CODE=1
export IGNORE_MGCAMB_CODE=1
export IGNORE_NAUTILUS_SAMPLER_CODE=1
export IGNORE_TENSIOMETER_CODE=1
export IGNORE_DARKEMULATOR_CODE=1
export IGNORE_EMULTRF_CODE=1
export PYTHON_VERSION=3.10
export MINICONDA_INSTALLATION=1
export MAKE_NUM_THREADS=$(nproc)
export OMP_NUM_THREADS=1
if [ -n "${MINICONDA_INSTALLATION}" ]; then
  source "${ROOTDIR:?}/installation_scripts/flags_miniconda_installation.sh"
  if [ $? -ne 0 ]; then return 1; fi
fi
source "${ROOTDIR:?}/installation_scripts/flags_derived.sh"
if [ $? -ne 0 ]; then return 1; fi
unset IGNORE_CPP_ARMA_INSTALLATION
SETOPT

# ── 5. Compile CosmoLike (direct build) ──
log "Step 5/8: Compile CosmoLike"
cd "$COCOA_INNER"
export ROOTDIR="$COCOA_INNER"

# ── 5a. Run setup_cocoa.sh to download cosmolike_core, simde, spdlog src ──
# WARNING: setup_cocoa.sh will exit≠0 because non-critical components
# (hyrec2, darkemulator, unxv tarballs) fail. That's expected.
# We only need: cosmolike_core ✅, simde ✅, spdlog source ✅
log "  5a. Running setup_cocoa.sh (downloading sources)..."
set +e
source setup_cocoa.sh 2>&1
SETUP_RC=$?
set -e
log "  setup_cocoa exit=$SETUP_RC"
if [ $SETUP_RC -ne 0 ]; then
    log "  WARNING: exit=$SETUP_RC (non-critical components failed, expected)"
fi

# Verify critical downloads
ECODEF="${ROOTDIR}/external_modules/code"
CORE="${ECODEF}/cosmolike_core"
if [ ! -d "$CORE/cosmolike" ]; then
    log "FATAL: cosmolike_core not found at $CORE"
    exit 1
fi
log "  cosmolike_core: OK"
if [ ! -d "${ECODEF}/simde" ]; then
    log "FATAL: simde not found at ${ECODEF}/simde"
    exit 1
fi
log "  simde: OK"

# ── 5b. Install spdlog + armadillo via conda ──
# ONLY these 2 — do NOT install lapack/openblas/arpack-ng/libgfortran!
# Those are already pinned in cocoapy310.yml; adding them causes conflicts.
log "  5b. Installing spdlog + armadillo via conda..."
set +e
conda install -y -q spdlog armadillo 2>&1 | tail -3
set -e
log "  spdlog + armadillo: OK"

# ── 5c. Create symlinks (replaces start_cocoa.sh lines 111-132) ──
log "  5c. Creating symlinks..."
ln -sfn "${CORE}/cfastpt"   "${ECODEF}/cfastpt"
ln -sfn "${CORE}/cosmolike" "${ECODEF}/cosmolike"
ln -sfn "${CORE}/log.c"     "${ECODEF}/log.c"
log "  symlinks: cfastpt, cosmolike, log.c → cosmolike_core/"

# ── 5d. Install headers + libs to .local/ ──
# setup_core_packages.sh downloaded headers to cocoa_installation_libraries/.
# unxv_core_packages.sh (which copies to .local/include/) failed. We copy manually.
log "  5d. Installing headers + libs to .local/..."
CCIL="${ROOTDIR}/../cocoa_installation_libraries"
mkdir -p "${ROOTDIR}/.local/include" "${ROOTDIR}/.local/lib"

# spdlog: headers from conda, lib symlinked
ln -sfn "${CONDA_PREFIX}/include/spdlog" "${ROOTDIR}/.local/include/spdlog"
if [ -f "${CONDA_PREFIX}/lib/libspdlog.a" ]; then
    cp "${CONDA_PREFIX}/lib/libspdlog.a" "${ROOTDIR}/.local/lib/"
elif [ -f "${CONDA_PREFIX}/lib/libspdlog.so" ]; then
    ln -sfn "${CONDA_PREFIX}/lib/libspdlog.so" "${ROOTDIR}/.local/lib/"
fi
log "  spdlog: conda → .local/"

# carma: header-only, from CCIL
if [ -d "${CCIL}/carma" ]; then
    cp -f "${CCIL}/carma/carma.h" "${ROOTDIR}/.local/include/" 2>/dev/null || true
    cp -rf "${CCIL}/carma/carma_bits" "${ROOTDIR}/.local/include/" 2>/dev/null || true
    log "  carma: CCIL → .local/include/"
else
    log "  WARNING: CCIL/carma not found"
fi

# armadillo: header-only (libs already in conda env from cocoapy310.yml)
# setup_core_packages.sh downloaded to CCIL as armadillo-<ver>/ directory
ARMA_DIR=$(find "${CCIL}" -maxdepth 1 -type d -name "armadillo*" 2>/dev/null | head -1)
if [ -n "$ARMA_DIR" ] && [ -d "$ARMA_DIR/include" ]; then
    cp -f "${ARMA_DIR}/include/armadillo" "${ROOTDIR}/.local/include/" 2>/dev/null || true
    cp -rf "${ARMA_DIR}/include/armadillo_bits" "${ROOTDIR}/.local/include/" 2>/dev/null || true
    log "  armadillo: CCIL → .local/include/"
elif [ -f "${CONDA_PREFIX}/include/armadillo" ]; then
    # Fallback: use conda's armadillo header if available
    ln -sfn "${CONDA_PREFIX}/include/armadillo" "${ROOTDIR}/.local/include/armadillo"
    [ -d "${CONDA_PREFIX}/include/armadillo_bits" ] && \\
        ln -sfn "${CONDA_PREFIX}/include/armadillo_bits" "${ROOTDIR}/.local/include/armadillo_bits"
    log "  armadillo: conda → .local/include/"
else
    log "  WARNING: armadillo header not found in CCIL or conda"
    log "  CCIL contents:"
    ls "$CCIL" 2>&1 | head -10
fi

# Verify critical headers
MISSING=0
for hdr in spdlog/spdlog.h carma.h; do
    if [ -f "${ROOTDIR}/.local/include/${hdr}" ]; then
        log "  ✓ ${hdr}"
    else
        log "  ✗ ${hdr} NOT FOUND"
        MISSING=1
    fi
done
if [ $MISSING -ne 0 ]; then
    log "  Contents of .local/include/:"
    ls -laR "${ROOTDIR}/.local/include/" 2>&1 | head -30
    log "  Contents of CCIL:"
    ls -la "$CCIL" 2>&1 | head -20
    log "FATAL: Missing required headers"
    exit 1
fi
log "  Headers OK"

# ── 5e. Compile CosmoLike directly ──
log "  5e. Compiling CosmoLike .so..."
export C_COMPILER="${CONDA_PREFIX}/bin/x86_64-conda-linux-gnu-cc"
export CXX_COMPILER="${CONDA_PREFIX}/bin/x86_64-conda-linux-gnu-g++"

# Verify compiler exists
if [ ! -f "$C_COMPILER" ]; then
    # Fallback to system gcc
    export C_COMPILER=gcc
    export CXX_COMPILER=g++
    log "  Using system gcc instead of conda compiler"
fi

cd "${ROOTDIR}/projects/des_y3/interface"

# Patch Makefile: replace hardcoded libspdlog.a path with dynamic -lspdlog
# (conda spdlog only provides .so, not .a)
sed -i 's|${ROOTDIR}/.local/lib/libspdlog.a|-lspdlog|g' MakefileCosmolike
log "  MakefileCosmolike patched: libspdlog.a → -lspdlog"

make -f MakefileCosmolike clean 2>/dev/null || true
make -j$(nproc) -f MakefileCosmolike all 2>&1

if ls "$DES_Y3_DIR/interface/"*.so > /dev/null 2>&1; then
    log "  CosmoLike .so OK"
    ls -la "$DES_Y3_DIR/interface/"*.so
else
    log "  FATAL: CosmoLike .so not found"
    exit 1
fi

# ── 6. Clone + Patch + Compile CLASS-EU inside conda ──
log "Step 6/7: CLASS-EU inside conda"
cd /content
if [ -d "$CLASS_CONDA" ]; then
    log "  Already present"
else
    git clone --depth 1 -b v3.3.4 https://github.com/lesgourg/class_public.git "$CLASS_CONDA" 2>&1 | tail -2
    log "  Cloned"
fi

cd "$CLASS_CONDA"

# Patch 1: parser.h buffer (CLASS v3.3.4 uses these names, NOT _MAX_NUMBER_)
sed -i 's/_LINE_LENGTH_MAX_ 1024/_LINE_LENGTH_MAX_ 8192/g' include/parser.h
sed -i 's/_ARGUMENT_LENGTH_MAX_ 1024/_ARGUMENT_LENGTH_MAX_ 8192/g' include/parser.h

# Patch 2: classy.pyx z_max_nonlinear
python3 -c "
with open('python/classy.pyx', 'r') as f:
    lines = f.readlines()
patched = False
for i in range(len(lines)):
    if 'get_pk_and_k_and_z() is trying to return P(k,z)' in lines[i]:
        j = i - 1
        while j >= 0 and 'if' not in lines[j]: j -= 1
        if j >= 0 and 'z_max_nonlinear' in lines[j]:
            for k in range(j, i+1):
                lines[k] = '            # PATCHED: ' + lines[k].lstrip()
            patched = True; break
if patched:
    with open('python/classy.pyx', 'w') as f: f.writelines(lines)
    print('[PATCH] classy.pyx z_max_nonlinear DISABLED')
else:
    print('[SKIP] z_max_nonlinear patch not needed or already applied')
"

# Patch 3: EU background/perturbations
if [ -n "$CLASS_EU_PATCH" ] && [ -f "$CLASS_EU_PATCH" ]; then
    python3 "$CLASS_EU_PATCH" "$CLASS_CONDA"
    log "  EU patches applied"
else
    log "  WARNING: CLASS_EU_PATCH not set, skipping EU patches"
fi

# Compile + install in conda env
# IMPORTANT: use system gcc AND system ld (conda ld has GLIBC_PRIVATE mismatch)
make clean > /dev/null 2>&1 || true
sed -i "s/^OPTFLAG.*=.*/OPTFLAG = -O3/" Makefile
PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin" make -j$(nproc) 2>&1 | tail -3
pip install --no-cache-dir --no-deps --force-reinstall . 2>&1 | tail -3

# Also build in-place so cobaya can find classy at path=/content/class_eu_conda
python3 setup.py build 2>&1 | tail -2

# CRITICAL: Replace Cocoa's classy in .local/site-packages/ with our EU version
# (start_cocoa.sh puts .local/ first in PYTHONPATH, shadowing conda's site-packages)
LOCAL_CLASSY="${ROOTDIR}/.local/lib/python3.10/site-packages/classy"
if [ -d "$LOCAL_CLASSY" ]; then
    # Copy the .so we just built (has parser.h _MAX_NUMBER_ 200 fix + EU patches)
    BUILD_SO=$(find . -name "_classy.cpython-310*.so" -path "*/build/*" 2>/dev/null | head -1)
    if [ -n "$BUILD_SO" ]; then
        cp -f "$BUILD_SO" "$LOCAL_CLASSY/"
        log "  Replaced .local classy .so with EU-patched binary"
    else
        log "  WARNING: build .so not found, trying pip install"
        pip install --target="${LOCAL_CLASSY}/.." --no-cache-dir --no-deps --force-reinstall . 2>&1 | tail -2
    fi
else
    log "  WARNING: .local/classy dir not found"
fi
log "  CLASS-EU compiled + installed"

# Smoke test
python3 -c "
from classy import Class
c = Class()
c.set({'output':'tCl', 'l_max_scalars':100,
       '100*theta_s': 1.0411,
       'omega_b': 0.02237, 'omega_cdm': 0.12,
       'eu_epsilon_ir': 0.04264, 'eu_z_trans': 5.986, 'eu_b': 0.5278,
       'Omega_Lambda': 0, 'w0_fld': -1.0, 'wa_fld': 0, 'cs2_fld': 1.0,
       'eu_has_ide_perturbations': 0,
       'N_ur': 2.0328, 'N_ncdm': 1, 'm_ncdm': 0.0589})
c.compute()
print(f'CLASS-EU conda OK | H0={c.h()*100:.2f}')
c.struct_cleanup(); c.empty()
"

# ── 7. Setup DES-Y3 likelihood symlink + shoes_lki ──
log "Step 7/7: Likelihood setup"
pip install -q euclidemu2 2>&1 | tail -1
log "  euclidemu2 installed"
COBAYA_LKL=$(python3 -c "import cobaya, os; print(os.path.join(os.path.dirname(cobaya.__file__), 'likelihoods'))")
SITE_PKGS=$(python3 -c "import site; print(site.getsitepackages()[0])")

# Symlink DES-Y3 likelihood into cobaya
ln -sf "$DES_Y3_DIR/likelihood" "$COBAYA_LKL/des_y3" 2>/dev/null || true

# Symlink the .so into site-packages so 'import cosmolike_des_y3_interface' works
ln -sf "$DES_Y3_DIR/interface/cosmolike_des_y3_interface.so" "$SITE_PKGS/" 2>/dev/null || true
log "  cosmolike .so → site-packages"

python3 -c "from cobaya.likelihoods.des_y3.combo_3x2pt import combo_3x2pt; print('DES-Y3 likelihood OK')"

# shoes_lki
mkdir -p /content/likelihoods
if [ -n "$SHOES_LKI_SRC" ] && [ -f "$SHOES_LKI_SRC" ]; then
    cp "$SHOES_LKI_SRC" /content/likelihoods/shoes_lki.py
    log "  shoes_lki.py copied"
fi

# Patch CosmoLike NaN guard
COSMOLIKE_BASE="$DES_Y3_DIR/likelihood/_cosmolike_prototype_base.py"
if [ -f "$COSMOLIKE_BASE" ]; then
    python3 << 'NANPATCH'
f = '/content/cocoa/Cocoa/projects/des_y3/likelihood/_cosmolike_prototype_base.py'
with open(f) as fh: content = fh.read()
old = """      elif self.non_linear_emul == 2:
        lnPNL = self.provider.get_Pk_interpolator(("delta_tot", "delta_tot"),
          nonlinear=True, 
          extrap_kmin=1e-6,
          extrap_kmax=2.5e2*self.accuracyboost).logP(self.z_interp_2D,
          np.power(10.0,self.log10k_interp_2D)).flatten(order='F')+np.log(h**3)"""
new = old + """
        # [EU-PATCH] Replace NaN/Inf in Halofit P_NL (z>10) with linear P(k)
        nan_mask = ~np.isfinite(lnPNL)
        if np.any(nan_mask):
          lnPNL[nan_mask] = lnPL[nan_mask]"""
if old in content and 'EU-PATCH' not in content:
    content = content.replace(old, new)
    with open(f, 'w') as fh: fh.write(content)
    print('[PATCH] CosmoLike NaN guard applied')
elif 'EU-PATCH' in content:
    print('[SKIP] NaN guard already present')
else:
    print('[WARN] Pattern not found')
NANPATCH
fi

log "ALL DONE"
echo "COCOA_SETUP_COMPLETE"

'''

with open('/content/setup_cocoa_conda.sh', 'w') as f:
    f.write(_setup_script)
print('  [OK] Script written (14703 bytes)')

# ── Ensure shoes_lki.py is available ──
os.makedirs('/content/likelihoods', exist_ok=True)
if os.path.exists('/content/shoes_lki.py'):
    shutil.copy2('/content/shoes_lki.py', '/content/likelihoods/shoes_lki.py')

# ── Execute ──
env = os.environ.copy()
env['CLASS_EU_PATCH'] = '/content/patch_class.py'
env['SHOES_LKI_SRC'] = '/content/shoes_lki.py'

print('\n  Running setup (miniforge + conda + Cocoa + CLASS-EU)...')
print('  This takes ~20-30 min. Progress logged below.\n')

r = subprocess.run(
    ['bash', '/content/setup_cocoa_conda.sh'],
    env=env, capture_output=True, text=True, timeout=3600
)

output = r.stdout
if len(output) > 5000:
    print('  [...truncated...]')
    print(output[-5000:])
else:
    print(output)

if r.returncode != 0 or 'COCOA_SETUP_COMPLETE' not in r.stdout:
    print(f'\n  [STDERR last 1000 chars]:\n{r.stderr[-1000:]}')
    raise RuntimeError(f'FATAL: Cocoa setup failed (exit={r.returncode})')

# ── Verify ──
so_files = glob.glob(f'{DES_Y3_DIR}/interface/*.so')
COCOA_READY = len(so_files) > 0

dt_cocoa = time.time() - t0_cocoa
print(f'\n{"=" * 60}')
assert COCOA_READY, f'FATAL: CosmoLike .so not found after {dt_cocoa:.0f}s'
print(f'  ✅ COCOA + CLASS-EU BUILD SUCCESS ({dt_cocoa:.0f}s)')
print(f'  CosmoLike .so: {so_files}')
print(f'  CLASS-EU conda: {CLASS_CONDA}')
print(f'  DES_Y3_DATA: {DES_Y3_DATA}')
print(f'{"=" * 60}')


---


In [ ]:
# ============================================================
# §3. CONFIGURATION VERIFICATION
# ============================================================
# Display the YAML that Cobaya actually used.
# This is the receipt — proof of exact run conditions.
# ============================================================
import yaml, glob

print('=' * 60)
print('RUN CONFIGURATION (from updated.yaml)')
print('=' * 60)

yaml_path = f'/content/{CHAIN_ROOT}.updated.yaml'
with open(yaml_path) as f:
    config = yaml.safe_load(f)

theory = config.get('theory', {}).get('classy', {})
extra_args = theory.get('extra_args', {})
likelihoods = list(config.get('likelihood', {}).keys())
sampler_cfg = config.get('sampler', {}).get('mcmc', {})

# EU perturbation flag — extracted from YAML, NOT hardcoded
eu_pert_flag = extra_args.get('eu_has_ide_perturbations', 'NOT_SET')

# EU params may be in params (fixed) or extra_args
params = config.get('params', {})
eu_param_vals = {}
for ep in ['eu_epsilon_ir', 'eu_z_trans', 'eu_b']:
    if ep in extra_args:
        eu_param_vals[ep] = extra_args[ep]
    elif ep in params and isinstance(params[ep], dict):
        eu_param_vals[ep] = params[ep].get('value', params[ep].get('ref', None))
    elif ep in params and not isinstance(params[ep], dict):
        eu_param_vals[ep] = params[ep]
    else:
        eu_param_vals[ep] = None
if eu_pert_flag == 0:
    eu_pert_label = 'OFF (background-only IDE, perturbation source terms \u0394k=0)'
elif eu_pert_flag == 1:
    eu_pert_label = 'ON (full IDE: background + perturbation source terms)'
else:
    eu_pert_label = f'UNKNOWN ({eu_pert_flag})'

print(f'\n  Theory: CLASS {theory.get("version", "?")}')
print(f'  EU params (FIXED):')
print(f'    \u03b5_IR     = {eu_param_vals.get("eu_epsilon_ir")}')
print(f'    z_trans  = {eu_param_vals.get("eu_z_trans")}')
print(f'    b        = {eu_param_vals.get("eu_b")}')
print(f'    \u03bb        = {extra_args.get("eu_lambda", "2/3 default")}')
print(f'  IDE perturbations: eu_has_ide_perturbations = {eu_pert_flag}')
print(f'    \u2192 {eu_pert_label}')
print(f'  Likelihoods ({len(likelihoods)}):')
for lik in likelihoods:
    print(f'    \u2022 {lik}')
print(f'  R-1 target: {sampler_cfg.get("Rminus1_stop", "?")}')
print(f'  Cobaya: {config.get("version", "?")}')

# Sampled vs derived
sampled = [p for p, v in params.items()
           if isinstance(v, dict) and 'prior' in v]
derived = [p for p, v in params.items()
           if isinstance(v, dict) and v.get('derived') is True]
print(f'\n  Sampled parameters ({len(sampled)}):')
for p in sampled:
    pr = params[p].get('prior', {})
    print(f'    {p}: {pr}')
print(f'\n  Derived parameters ({len(derived)}):')
for p in derived:
    print(f'    {p}')


---


In [ ]:
# ============================================================
# §4. CONVERGENCE DIAGNOSTICS + TRACE PLOTS
# ============================================================

print('=' * 60)
print('CONVERGENCE DIAGNOSTICS')
print('=' * 60)

chain_files = sorted(glob.glob(f'/content/{CHAIN_ROOT}.*.txt'))
assert len(chain_files) == N_CHAINS, f'Expected {N_CHAINS} chains, got {len(chain_files)}. No fallback.'

# Read header
with open(chain_files[0]) as f:
    header = f.readline().strip().lstrip('#').split()

# Load chains with 30% burn-in
chains_raw = []
for cf in chain_files:
    data = np.loadtxt(cf)
    burn = int(0.3 * len(data))
    chains_raw.append(data[burn:])
    print(f'  {os.path.basename(cf)}: {len(data)} rows, burn={burn}, kept={len(data)-burn}')

# Gelman-Rubin R-1
def gelman_rubin(chains, col_idx, weight_idx=0):
    chain_means, chain_vars, chain_n = [], [], []
    for c in chains:
        w = c[:, weight_idx]
        x = c[:, col_idx]
        n = np.sum(w)
        mean = np.average(x, weights=w)
        var = np.average((x - mean)**2, weights=w)
        chain_means.append(mean)
        chain_vars.append(var)
        chain_n.append(n)
    m = len(chains)
    grand_mean = np.mean(chain_means)
    n_avg = np.mean(chain_n)
    B = n_avg / (m - 1) * sum((mu - grand_mean)**2 for mu in chain_means)
    W = np.mean(chain_vars)
    if W < 1e-30:
        return np.nan
    V = (1 - 1/n_avg) * W + B / n_avg
    return V / W - 1

print(f'\n{"Parameter":<25} {"R-1":>10} {"Status":>8} {"Mean":>14} {"Std":>12}')
print('-' * 72)

all_data = np.vstack(chains_raw)
all_w = all_data[:, 0]

skip = {'weight', 'minuslogpost', 'minuslogprior', 'minuslogprior__0', 'chi2'}
r1_results = {}

for j, name in enumerate(header):
    if name in skip or name.startswith('chi2__'):
        continue
    r1 = gelman_rubin(chains_raw, j)
    mean = np.average(all_data[:, j], weights=all_w)
    std = np.sqrt(np.average((all_data[:, j] - mean)**2, weights=all_w))
    r1_results[name] = {'R-1': r1, 'mean': mean, 'std': std}
    if np.isnan(r1):
        status = '(const)'
    elif r1 < 0.01:
        status = '\u2705'
    elif r1 < 0.05:
        status = '\u26a0\ufe0f'
    else:
        status = '\u274c'
    print(f'  {name:<23} {r1:10.4f} {status:>8} {mean:14.6f} {std:12.6f}')

sampled_r1 = {k: v for k, v in r1_results.items() if not np.isnan(v['R-1'])}
converged = sum(1 for v in sampled_r1.values() if v['R-1'] < 0.01)
r1_max = max(v['R-1'] for v in sampled_r1.values())
total_weighted = int(np.sum(all_w))

print(f'\n  Converged: {converged}/{len(sampled_r1)} (R-1 < 0.01)')
print(f'  Worst R-1: {r1_max:.4f}')
print(f'  Total weighted samples: {total_weighted:,}')

# ── Effective Sample Size (ESS) via GetDist ──
from getdist import MCSamples
print(f'\n  Effective Sample Size (ESS):')
combined_gd = MCSamples(
    samples=all_data[:, 2:], weights=all_data[:, 0], loglikes=all_data[:, 1],
    names=header[2:], labels=header[2:]
)
ess_key = ['omega_b', 'omega_cdm', 'H0', 'tau_reio', 'logA', 'n_s', 'sigma8', 'S8']
for p in ess_key:
    if p in header[2:]:
        idx = header[2:].index(p)
        try:
            ess = combined_gd.getEffectiveSamplesForParamIndex(idx)
            print(f'    {p:<16} ESS = {ess:,.0f}')
        except:
            pass

# ── Trace Plots ──
import matplotlib.pyplot as plt
os.makedirs('figures', exist_ok=True)

trace_params = ['H0', 'omega_cdm', 'sigma8', 'S8', 'tau_reio', 'n_s']
trace_available = [p for p in trace_params if p in header]

fig, axes = plt.subplots(len(trace_available), 1, figsize=(14, 3 * len(trace_available)), sharex=True)
if len(trace_available) == 1:
    axes = [axes]

colors = plt.cm.tab10(np.linspace(0, 1, N_CHAINS))

for ax, pname in zip(axes, trace_available):
    col_idx = header.index(pname)
    for ci, chain in enumerate(chains_raw):
        # Thin for plotting (every 10th sample)
        step = max(1, len(chain) // 2000)
        samples = chain[::step, col_idx]
        ax.plot(samples, alpha=0.5, linewidth=0.5, color=colors[ci], label=f'Chain {ci+1}' if pname == trace_available[0] else None)
    mean_val = r1_results[pname]['mean']
    ax.axhline(mean_val, color='red', linewidth=1.5, linestyle='--', alpha=0.8)
    ax.set_ylabel(pname, fontsize=12)
    r1_val = r1_results[pname]['R-1']
    ax.set_title(f'{pname}   (R-1 = {r1_val:.4f})', fontsize=11, loc='right')

axes[0].legend(loc='upper right', ncol=4, fontsize=8)
axes[-1].set_xlabel('Sample (post burn-in, thinned)', fontsize=12)
fig.suptitle('Trace Plots \u2014 8 Independent MPI Chains', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig_NB05_D2_trace_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] Trace plots saved')


---


In [ ]:
# ============================================================
# §5. CORNER PLOTS (GetDist)
# ============================================================

from getdist import MCSamples, plots
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({
    'font.family': 'serif', 'font.size': 11,
    'axes.labelsize': 12, 'figure.facecolor': 'white'
})
os.makedirs('figures', exist_ok=True)

print('=' * 60)
print('CORNER PLOTS')
print('=' * 60)

# Combined sample
all_post = np.vstack([c[int(0.3*len(c)):] for c in [np.loadtxt(cf) for cf in chain_files]])
combined = MCSamples(
    samples=all_post[:, 2:], weights=all_post[:, 0], loglikes=all_post[:, 1],
    names=header[2:], labels=header[2:],
    name_tag='D2 Combined', label='EU D2'
)

# -- Cosmological triangle --
cosmo_params = ['omega_b', 'omega_cdm', 'H0', 'tau_reio', 'logA', 'n_s', 'sigma8', 'S8']
cosmo_available = [p for p in cosmo_params if p in header[2:]]

g = plots.get_subplot_plotter(width_inch=14)
g.settings.axes_fontsize = 10
g.settings.axes_labelsize = 12
g.settings.title_limit_fontsize = 10
g.triangle_plot(combined, cosmo_available, filled=True,
                contour_colors=['#2196F3'], title_limit=1)
plt.subplots_adjust(hspace=0.08, wspace=0.08)
plt.savefig('figures/fig_NB05_D2_corner_cosmo.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB05_D2_corner_cosmo.pdf', bbox_inches='tight')
plt.show()
print('[OK] Cosmological corner plot')

# -- EU-derived --
eu_params = ['H0', 'H0_LKI', 'sigma8', 'S8', 'Omega_m', 'rdrag']
eu_available = [p for p in eu_params if p in header[2:]]

g2 = plots.get_subplot_plotter(width_inch=12)
g2.settings.axes_fontsize = 11
g2.settings.axes_labelsize = 13
g2.settings.title_limit_fontsize = 11
g2.triangle_plot(combined, eu_available, filled=True,
                 contour_colors=['#4CAF50'], title_limit=1)
plt.subplots_adjust(hspace=0.08, wspace=0.08)
plt.savefig('figures/fig_NB05_D2_corner_eu.png', dpi=150, bbox_inches='tight')
plt.savefig('figures/fig_NB05_D2_corner_eu.pdf', bbox_inches='tight')
plt.show()
print('[OK] EU-derived corner plot')

# -- 1D posteriors --
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
key_params = ['H0', 'H0_LKI', 'omega_cdm', 'omega_b',
              'sigma8', 'S8', 'Omega_m', 'n_s']
for ax, pname in zip(axes.flat, key_params):
    if pname in header[2:]:
        idx = header[2:].index(pname)
        vals = all_post[:, idx + 2]
        weights = all_post[:, 0]
        ax.hist(vals, bins=60, weights=weights, density=True,
                color='#2196F3', alpha=0.7, edgecolor='none')
        mean = np.average(vals, weights=weights)
        ax.axvline(mean, color='red', linewidth=1.5, linestyle='--')
        ax.set_title(pname, fontsize=12)
        ax.set_yticks([])
plt.suptitle('C2 Baseline \u2014 1D Posteriors', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/fig_NB05_D2_1d_posteriors.png', dpi=150, bbox_inches='tight')
plt.show()
print('[OK] 1D posteriors')


---


In [ ]:
# ============================================================
# §6. PARAMETER TABLE + BEST-FIT
# ============================================================

print('=' * 60)
print('PARAMETER TABLE')
print('=' * 60)

bf_idx = np.argmin(all_data[:, header.index('minuslogpost')])
bestfit = all_data[bf_idx]

cosmo_table = ['omega_b', 'omega_cdm', 'theta_s_100', 'tau_reio',
               'logA', 'n_s', 'A_planck']
derived_table = ['H0', 'sigma8', 'S8', 'Omega_m', 'rdrag',
                 'H0_LKI', 'fcdm_z0', 'I_GKI']

print(f'\n{"Parameter":<18} {"Mean":>12} {"Std":>12} {"Best-fit":>12} {"R-1":>8}')
print('-' * 65)
print('  --- Sampled ---')
for p in cosmo_table:
    if p in r1_results:
        r = r1_results[p]
        bf = bestfit[header.index(p)] if p in header else np.nan
        r1_str = f'{r["R-1"]:.4f}' if not np.isnan(r['R-1']) else 'const'
        print(f'  {p:<16} {r["mean"]:12.6f} {r["std"]:12.6f} {bf:12.6f} {r1_str:>8}')

print('  --- Derived ---')
for p in derived_table:
    if p in r1_results:
        r = r1_results[p]
        bf = bestfit[header.index(p)] if p in header else np.nan
        r1_str = f'{r["R-1"]:.4f}' if not np.isnan(r['R-1']) else 'const'
        print(f'  {p:<16} {r["mean"]:12.6f} {r["std"]:12.6f} {bf:12.6f} {r1_str:>8}')

# Chi2 breakdown
print(f'\n  Best-fit chi2:')
chi2_keys = [h for h in header if h.startswith('chi2__')]
for k in chi2_keys:
    val = bestfit[header.index(k)]
    print(f'    {k.replace("chi2__", ""):<45} {val:10.3f}')
total_from_col = bestfit[header.index('chi2')] if 'chi2' in header else 0
print(f'    {"TOTAL":<45} {total_from_col:10.3f}')


---


In [ ]:
# ============================================================
# §7. CROSS-CHECK — Unified evaluation at best-fit (ALL likelihoods)
# ============================================================
# Single evaluation inside conda env with ALL 8 likelihoods.
# Self-contained: writes crosscheck script, then executes in conda.
# Requires: §2.5 completed (Cocoa + CLASS-EU in conda)
# Time: ~2-5 min
# ============================================================

import json as _json, subprocess, os
import numpy as np

print('=' * 60)
print('CROSS-CHECK — Unified likelihood evaluation at best-fit')
print('=' * 60)

CONDA_DIR = '/content/miniforge3'
COCOA_INNER = '/content/cocoa/Cocoa'
CLASS_CONDA = '/content/class_eu_conda'
DES_Y3_DATA = f'{COCOA_INNER}/projects/des_y3/data'
CROSSCHECK_SCRIPT = '/content/crosscheck_full.py'
RESULT_JSON = '/content/crosscheck_result.json'
BESTFIT_JSON = '/content/bestfit_params.json'

try:
    # ── Write crosscheck script to disk ──
    print('\n  [0/3] Writing crosscheck_full.py...')
    _xcheck_code = '''\
#!/usr/bin/env python3
"""
Cross-check: Evaluate ALL likelihoods at best-fit point.
Runs INSIDE conda env with CosmoLike compiled.
Called from NB05_MCMC_D2.ipynb §7 via subprocess.

Usage:
  python3 crosscheck_full.py \
    --yaml /content/eu_NB05D2.updated.yaml \
    --bestfit /content/bestfit_params.json \
    --class-dir /content/class_eu_conda \
    --packages /content/packages \
    --des-data /content/cocoa/Cocoa/projects/des_y3/data \
    --output /content/crosscheck_result.json

REQUIRED: Caller must set LD_LIBRARY_PATH to include:
  - <cocoa>/Cocoa/.local/lib
  - $CONDA_PREFIX/lib
"""

import argparse, json, sys, os
import numpy as np


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--yaml', required=True)
    parser.add_argument('--bestfit', required=True)
    parser.add_argument('--class-dir', required=True)
    parser.add_argument('--packages', required=True)
    parser.add_argument('--des-data', required=True)
    parser.add_argument('--output', required=True)
    parser.add_argument('--shoes-dir', default='/content/likelihoods')
    args = parser.parse_args()

    # Add shoes_lki to path
    if args.shoes_dir not in sys.path:
        sys.path.insert(0, args.shoes_dir)

    # NOTE: classy lives in Cocoa/.local/site-packages/ (pip installed our EU version there)
    # Do NOT remove it — that's the only copy. Just don't set path= so cobaya won't complain.

    import yaml
    from cobaya.run import run as cobaya_run

    # Load YAML
    with open(args.yaml) as f:
        info = yaml.safe_load(f)

    # Load best-fit params
    with open(args.bestfit) as f:
        bf = json.load(f)

    # Configure for evaluate mode
    info['sampler'] = {'evaluate': None}
    info['output'] = '/content/verify_d2_full'
    info['force'] = True
    info['packages_path'] = args.packages

    # Let cobaya find classy from site-packages (pip installed our EU version)
    # Do NOT set path= as cobaya's path validation conflicts with pip installs
    if 'classy' in info.get('theory', {}):
        info['theory']['classy'].pop('path', None)

    # Point DES-Y3 data
    if 'des_y3.combo_3x2pt' in info.get('likelihood', {}):
        info['likelihood']['des_y3.combo_3x2pt']['path'] = args.des_data

    # Fix all sampled params to best-fit values
    for p, v in list(info['params'].items()):
        if isinstance(v, dict) and 'prior' in v:
            if p in bf:
                info['params'][p] = {'value': bf[p]}

    # NOTE: Do NOT remove 'As' — DES-Y3 needs it (derived from logA via lambda)

    # Remove eu_derived if present (computed by shoes_lki)
    info['likelihood'].pop('eu_derived.EU_Derived', None)

    # Fix shoes_lki python_path to absolute
    shoes_key = 'shoes_lki.SH0ES_LKI'
    if shoes_key in info.get('likelihood', {}):
        lik_cfg = info['likelihood'][shoes_key]
        if isinstance(lik_cfg, dict) and lik_cfg.get('python_path') == 'likelihoods':
            lik_cfg['python_path'] = args.shoes_dir

    print(f'[crosscheck] Evaluating with {len(info["likelihood"])} likelihoods...')
    for k in info['likelihood']:
        print(f'  - {k}')

    # Run evaluation
    updated, sampler = cobaya_run(info)

    # Extract results
    products = sampler.products()
    sample = products.get('sample', products.get('minimum', None))

    result = {'chi2': {}, 'derived': {}, 'logpost': None}
    if sample is not None:
        if hasattr(sample, 'data'):
            row = sample.data.iloc[0].to_dict()
        elif hasattr(sample, 'iloc'):
            row = sample.iloc[0].to_dict()
        else:
            row = {}

        # Cobaya aggregate keys (chi2__BAO, chi2__CMB, chi2__SN) double-count
        # individual likelihoods — skip them
        AGGREGATE_KEYS = {'chi2__BAO', 'chi2__CMB', 'chi2__SN'}
        for k, v in row.items():
            k_str = str(k)
            if k_str.startswith('chi2__') and k_str not in AGGREGATE_KEYS:
                result['chi2'][k_str] = float(v)
            elif k_str == 'minuslogpost':
                result['logpost'] = float(v)
            elif k_str in ['H0', 'sigma8', 'S8', 'Omega_m', 'rdrag',
                           'H0_LKI', 'I_GKI', 'fcdm_z0']:
                result['derived'][k_str] = float(v)

    result['chi2_total'] = sum(result['chi2'].values())
    result['status'] = 'OK'

    with open(args.output, 'w') as f:
        json.dump(result, f, indent=2)

    print(f'[crosscheck] chi2_total = {result["chi2_total"]:.3f}')
    print('CROSSCHECK_COMPLETE')


if __name__ == '__main__':
    main()

'''
    with open(CROSSCHECK_SCRIPT, 'w') as f:
        f.write(_xcheck_code)
    print(f'    Written ({len(_xcheck_code)} bytes)')

    # ── Export best-fit params as JSON ──
    print('\n  [1/3] Exporting best-fit params...')
    bf_dict = {}
    for p in header:
        if not p.startswith('chi2__') and not p.startswith('minus') and p != 'weight':
            idx = header.index(p)
            if idx < len(bestfit):
                bf_dict[p] = float(bestfit[idx])
    with open(BESTFIT_JSON, 'w') as f:
        _json.dump(bf_dict, f, indent=2)
    print(f'    {len(bf_dict)} params exported')

    # ── Run crosscheck inside conda ──
    print('\n  [2/3] Running cross-check inside conda (ALL likelihoods)...')

    cmd = (
        f'export PATH="{CONDA_DIR}/bin:$PATH" && '
        f'eval "$({CONDA_DIR}/bin/conda shell.bash hook)" && '
        f'conda activate cocoa && '
        f'cd {COCOA_INNER} && '
        f'source start_cocoa.sh > /dev/null 2>&1; '
        f'export LD_LIBRARY_PATH="{COCOA_INNER}/.local/lib:$CONDA_PREFIX/lib:$LD_LIBRARY_PATH" && '
        f'export OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 && '
        f'python3 {CROSSCHECK_SCRIPT} '
        f'  --yaml {yaml_path} '
        f'  --bestfit {BESTFIT_JSON} '
        f'  --class-dir {CLASS_CONDA} '
        f'  --packages {PACKAGES_PATH} '
        f'  --des-data {DES_Y3_DATA} '
        f'  --shoes-dir /content/likelihoods '
        f'  --output {RESULT_JSON}'
    )

    r = subprocess.run(
        ['bash', '-c', cmd],
        capture_output=True, text=True, timeout=600
    )

    if r.stdout:
        print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)

    if 'CROSSCHECK_COMPLETE' not in r.stdout:
        print(f'\n  [STDERR first 3000 chars]:\n{r.stderr[:3000]}')
        raise RuntimeError('Cross-check subprocess failed')

    # ── Read results and compare ──
    print('\n  [3/3] Comparing chain vs live evaluation...')
    with open(RESULT_JSON) as f:
        xcheck = _json.load(f)

    live_chi2 = xcheck['chi2']
    live_total = xcheck['chi2_total']
    chain_total = float(total_from_col)
    diff = abs(chain_total - live_total)

    print(f'\n  {"=" * 50}')
    print(f'  χ² COMPARISON')
    print(f'  {"=" * 50}')
    print(f'  Chain best-fit χ²  = {chain_total:.3f}')
    print(f'  Live evaluation χ² = {live_total:.3f}')
    print(f'  Difference:         {diff:.3f}')

    if diff < 1.0:
        print(f'  ✅ MATCH — Chains verified')
    elif diff < 5.0:
        print(f'  ⚠️ CLOSE — Minor numerical difference')
    else:
        print(f'  ❌ MISMATCH')

    # Per-likelihood breakdown
    print(f'\n  Per-likelihood:')
    print(f'    {"Likelihood":<45} {"Chain":>10} {"Live":>10} {"Diff":>8}')
    print(f'    {"-"*75}')
    for col in sorted(live_chi2.keys()):
        if col in header:
            short = col.replace('chi2__', '')
            chain_val = float(bestfit[header.index(col)])
            live_val = live_chi2[col]
            d = abs(chain_val - live_val)
            status = '✅' if d < 0.5 else '⚠️'
            print(f'    {status} {short:<43} {chain_val:10.3f} {live_val:10.3f} {d:8.4f}')

    chain_logpost = float(bestfit[header.index('minuslogpost')])
    print(f'\n  Chain -logpost = {chain_logpost:.3f}')
    if xcheck.get('logpost') is not None:
        print(f'  Live  -logpost = {xcheck["logpost"]:.3f}')

    crosscheck_passed = diff < 5.0

except Exception as e:
    print(f'\n  [ERROR] Cross-check failed: {e}')
    import traceback
    traceback.print_exc()
    crosscheck_passed = False


---


In [ ]:
# ============================================================
# §8. EXPORT — NB05_D2_results.json
# ============================================================
import json
import os as _os

# ── Read Cobaya checkpoint for official R-1 ──
cobaya_r1_raw = None
for _cp_try in [f'/content/{CHAIN_ROOT}.checkpoint', f'{CHAIN_ROOT}.checkpoint']:
    if _os.path.exists(_cp_try):
        with open(_cp_try) as _cpf:
            for _line in _cpf:
                if 'Rminus1_last' in _line:
                    cobaya_r1_raw = float(_line.split(':')[1].strip())
                    print(f'  [CHECKPOINT] R-1 cobaya = {cobaya_r1_raw:.6f}')
        break
if cobaya_r1_raw is None:
    print('  [WARN] No .checkpoint file — using post-burnin R-1 as cobaya value')
    cobaya_r1_raw = r1_max

print('=' * 60)
print('EXPORT')
print('=' * 60)

def get_mean(name):
    return float(r1_results[name]['mean'])

def get_std(name):
    return float(r1_results[name]['std'])

def get_bf(name):
    if name in header and header.index(name) < len(bestfit):
        return float(bestfit[header.index(name)])
    elif name in r1_results:
        return r1_results[name]['mean']  # post-hoc: bestfit = mean
    return float('nan')

def get_r1_val(name):
    r = r1_results.get(name, {}).get('R-1', np.nan)
    return None if np.isnan(r) else round(float(r), 6)

# Confidence intervals via weighted percentiles
def weighted_percentile(data, weights, percentiles):
    sorted_idx = np.argsort(data)
    sorted_data = data[sorted_idx]
    sorted_weights = weights[sorted_idx]
    cumsum = np.cumsum(sorted_weights)
    cumsum /= cumsum[-1]
    return np.interp(percentiles, cumsum, sorted_data)

def get_ci(name, levels=[0.025, 0.16, 0.50, 0.84, 0.975]):
    if name not in header:
        # Post-hoc constant (e.g. fcdm_z0, I_GKI) — no distribution
        m = r1_results[name]['mean']
        return {'median': m, '68_lower': m, '68_upper': m, '95_lower': m, '95_upper': m}
    idx = header.index(name)
    vals = all_data[:, idx]
    pcts = weighted_percentile(vals, all_w, levels)
    return {
        'median': round(float(pcts[2]), 6),
        '68_lower': round(float(pcts[1]), 6),
        '68_upper': round(float(pcts[3]), 6),
        '95_lower': round(float(pcts[0]), 6),
        '95_upper': round(float(pcts[4]), 6),
    }

# Chi2 bestfit dict
chi2_bf = {}
for k in header:
    if k.startswith('chi2__') and k != 'chi2':
        chi2_bf[k.replace('chi2__', '')] = get_bf(k)
chi2_bf['total'] = float(total_from_col)

# N_DATA from config likelihoods (verified against Cobaya log)
# CamSpec prints "Number of data points: 9915" in the log.
# lowl, lensing, BAO, SNe from official Cobaya docs.
N_DATA_PER_LIK = {}
for lik_name in config.get('likelihood', {}).keys():
    if 'CamSpec' in lik_name and 'TTTEEE' in lik_name:
        N_DATA_PER_LIK[lik_name] = 9915   # from Cobaya log
    elif lik_name == 'planck_2018_lowl.TT':
        N_DATA_PER_LIK[lik_name] = 28     # ell 2-29
    elif lik_name == 'planck_2018_lowl.EE':
        N_DATA_PER_LIK[lik_name] = 396    # SimAll bins
    elif lik_name == 'planck_2018_lensing.clik':
        N_DATA_PER_LIK[lik_name] = 9      # phi bandpowers
    elif lik_name == 'bao.desi_dr2':
        N_DATA_PER_LIK[lik_name] = 12     # DESI DR2 data points
    elif lik_name == 'sn.pantheonplus':
        N_DATA_PER_LIK[lik_name] = 1701   # PantheonPlus SNe
    elif 'des_y3' in lik_name:
        N_DATA_PER_LIK[lik_name] = 462    # DES-Y3 3x2pt data vector
    # eu_derived.EU_Derived: logp=0, not a constraint

n_data_total = sum(N_DATA_PER_LIK.values())
n_sampled = len([p for p, v in config['params'].items()
                 if isinstance(v, dict) and 'prior' in v])
n_dof = n_data_total - n_sampled

# Post-hoc derived params if not in chains
# Void boost computed from formula (not hardcoded)
# Wu & Huterer 2017 Eq. 7 + Marra+ 2013 Θ correction (PRL 110, 241305)
H0_GKI_BASE = 68.90             # NB02 analytic H0_GKI (2026-06-15 audit)
_delta_obs_KBC = -0.46           # Keenan, Barger & Cowie (2013)
_f_growth_ref = 0.5140           # Riccati ODE at Omega_m_EU (NB02 §4.1, exact)
_delta_true_ref = _delta_obs_KBC / (1.0 + _f_growth_ref)  # RSD correction (Haslbauer+ 2020)
_Theta_ref = 1.0 - 0.0882 * _delta_true_ref - 0.123 * np.sin(_delta_true_ref) / (1.29 + _delta_true_ref)
dH0_VOID_BASE = -(1.0/3.0) * _delta_true_ref * _f_growth_ref * _Theta_ref * H0_GKI_BASE

if 'H0' in r1_results:  # FORCE recompute H0_LKI with Theta (overrides chain value)
    h0_col = header.index('H0')
    h0_vals = all_data[:, h0_col]
    h0_lki_vals = h0_vals + dH0_VOID_BASE * (h0_vals / H0_GKI_BASE)
    h0_lki_mean = np.average(h0_lki_vals, weights=all_w)
    h0_lki_std = np.sqrt(np.average((h0_lki_vals - h0_lki_mean)**2, weights=all_w))
    r1_results['H0_LKI'] = {'R-1': r1_results['H0']['R-1'], 'mean': h0_lki_mean, 'std': h0_lki_std}
    if 'H0_LKI' in header:
        _lki_idx = header.index('H0_LKI')
        all_data[:, _lki_idx] = h0_lki_vals
    else:
        all_data = np.column_stack([all_data, h0_lki_vals])
        header.append('H0_LKI')
    bf_h0 = float(bestfit[header.index('H0') if 'H0' in header[:len(bestfit)] else 0])
    bf_h0_lki = bf_h0 + dH0_VOID_BASE * (bf_h0 / H0_GKI_BASE)
    if 'H0_LKI' in header and header.index('H0_LKI') < len(bestfit):
        bestfit[header.index('H0_LKI')] = bf_h0_lki  # overwrite
    else:
        bestfit = np.append(bestfit, bf_h0_lki)
    print(f'  [COMPUTED] H0_LKI = {h0_lki_mean:.3f} ± {h0_lki_std:.3f} (post-hoc from H0)')

# Helper to extract float from eu_param_vals (may be dict with 'value' key)
def _eu_float(key, default):
    v = eu_param_vals.get(key, default)
    if isinstance(v, dict):
        return float(v.get('value', v.get('ref', v.get('mean', default))))
    return float(v) if v is not None else default

if True:  # FORCE recalculate with correct formula (2026-06-15 fix)
    # Correct formula: fcdm = exp(-λ · I_GKI) via numerical integration
    eps = _eu_float('eu_epsilon_ir', 0.04264)
    zt = _eu_float('eu_z_trans', 5.986)
    bb = _eu_float('eu_b', 0.52778)
    lam = 2.0/3.0
    _z_grid = np.linspace(0, 50, 2000)
    _eps_z = eps / (1.0 + ((1.0 + _z_grid) / (1.0 + zt))**(1.0 / bb))
    _eps_z[_z_grid > zt] = 0.0
    _trapz_fn = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
    _I_GKI_val = _trapz_fn(_eps_z / (1.0 + _z_grid), _z_grid)
    fcdm = np.exp(-lam * _I_GKI_val)
    r1_results['fcdm_z0'] = {'R-1': float('nan'), 'mean': fcdm, 'std': 0.0}
    print(f'  [COMPUTED] fcdm_z0 = {fcdm:.6f} (exp(-λ·I_GKI), numerical)')

if True:  # FORCE recalculate with correct formula (2026-06-15 fix)
    eps = _eu_float('eu_epsilon_ir', 0.04264)
    zt = _eu_float('eu_z_trans', 5.986)
    bb = _eu_float('eu_b', 0.52778)
    lam = 2.0/3.0
    _z_grid2 = np.linspace(0, 50, 2000)
    _eps_z2 = eps / (1.0 + ((1.0 + _z_grid2) / (1.0 + zt))**(1.0 / bb))
    _eps_z2[_z_grid2 > zt] = 0.0
    _trapz_fn2 = getattr(np, 'trapezoid', getattr(np, 'trapz', None))
    I_val = _trapz_fn2(_eps_z2 / (1.0 + _z_grid2), _z_grid2)
    r1_results['I_GKI'] = {'R-1': float('nan'), 'mean': I_val, 'std': 0.0}
    print(f'  [COMPUTED] I_GKI = {I_val:.6f} (numerical ∫ε/(1+z)dz)')

# Tensions
H0_SHOES, H0_SHOES_ERR = 73.17, 0.86
S8_DES, S8_DES_ERR = 0.776, 0.017

# Build param entries with full info
def param_entry(name):
    entry = {
        'mean': round(get_mean(name), 6),
        'std': round(get_std(name), 6),
        'bestfit': round(get_bf(name), 6),
        'R-1': get_r1_val(name),
    }
    entry.update(get_ci(name))
    return entry

# ESS from GetDist
ess_dict = {}
for p in header[2:]:
    idx = header[2:].index(p)
    try:
        ess = combined_gd.getEffectiveSamplesForParamIndex(idx)
        ess_dict[p] = int(ess)
    except:
        pass

# Perturbation flag from YAML
eu_pert_flag = extra_args.get('eu_has_ide_perturbations', 'NOT_SET')

# Full results
results = {
    '_metadata': {
        'notebook': 'NB05_MCMC_D2',
        'run_label': f'D2 ({CHAIN_ROOT}, eu_has_ide_perturbations={eu_pert_flag})',
        'run_description': eu_pert_label,
        'date': str(np.datetime64('now')),
        'source': f'{CHAIN_ROOT} ({N_CHAINS} chains)',
        'convergence': {
            'R-1_max_post_burnin': round(r1_max, 6),
            'R-1_max_cobaya': round(float(cobaya_r1_raw), 6) if cobaya_r1_raw is not None else None,
            'target': float(sampler_cfg.get('Rminus1_stop', 0.01)),
            'status': ('FULLY_CONVERGED' if cobaya_r1_raw is not None and cobaya_r1_raw < 0.01 else 'CONVERGED' if cobaya_r1_raw is not None and cobaya_r1_raw < 0.03 else 'NOT_CONVERGED'),
        },
        'samples_weighted': total_weighted,
        'effective_sample_size': ess_dict,
        'n_chains': N_CHAINS,
        'burn_in': '30%',
        'crosscheck': 'PASSED' if crosscheck_passed else ('SKIPPED' if crosscheck_passed is None else 'FAILED'),
        'software': {
            'cobaya': cobaya.__version__,
            'getdist': getdist.__version__,
            'class': config.get('theory', {}).get('classy', {}).get('version', 'v3.3.4'),
            'eu_patch': 'vf1.3',
        },
        'likelihoods': likelihoods,
        'n_data_per_likelihood': N_DATA_PER_LIK,
        'n_data_total': n_data_total,
        'n_sampled_params': n_sampled,
        'n_dof': n_dof,
        'chi2_per_dof': round(float(total_from_col) / n_dof, 4),
    },
    'cosmological_params': {
        p: param_entry(p)
        for p in ['omega_b', 'omega_cdm', 'theta_s_100', 'tau_reio', 'logA', 'n_s', 'A_planck']
        if p in r1_results
    },
    'derived_params': {},
    'convergence_per_param': {
        name: get_r1_val(name)
        for name in r1_results
        if get_r1_val(name) is not None
    },
    'eu_params_fixed': {
        'eps_IR': eu_param_vals.get('eu_epsilon_ir'),
        'z_trans': eu_param_vals.get('eu_z_trans'),
        'b': eu_param_vals.get('eu_b'),
        'lambda': extra_args.get('eu_lambda', 0.66667),
        'eu_has_ide_perturbations': eu_pert_flag,
    },
    'chi2_bestfit': chi2_bf,
    'chi2_per_dof': round(float(total_from_col) / n_dof, 4),
    'tensions': {
        'H0_LKI_vs_SH0ES_sigma': round(abs(get_mean('H0_LKI') - H0_SHOES) / H0_SHOES_ERR, 2),
        'H0_GKI_vs_SH0ES_sigma': round(abs(get_mean('H0') - H0_SHOES) / H0_SHOES_ERR, 2),
        'S8_vs_DES_sigma': round(abs(get_mean('S8') - S8_DES) / S8_DES_ERR, 2),
    },
}

# Derived params with full stats
for p in ['H0', 'sigma8', 'S8', 'Omega_m', 'fcdm_z0', 'H0_LKI', 'I_GKI', 'rdrag']:
    if p in r1_results:
        entry = param_entry(p)
        if p == 'rdrag':
            entry['unit'] = 'Mpc'
        results['derived_params'][p] = entry

with open('NB05_D2_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=float)
print(f'  [SAVED] NB05_D2_results.json')

# Summary
print(f'\n  === Key Results ===')
print(f'  IDE perturbations: eu_has_ide_perturbations = {eu_pert_flag}')
print(f'    \u2192 {eu_pert_label}')
print(f'  H0     = {get_mean("H0"):.3f} \u00b1 {get_std("H0"):.3f}  [{get_ci("H0")["95_lower"]:.2f}, {get_ci("H0")["95_upper"]:.2f}] (95% CL)')
print(f'  H0_LKI = {get_mean("H0_LKI"):.3f} \u00b1 {get_std("H0_LKI"):.3f}  [{get_ci("H0_LKI")["95_lower"]:.2f}, {get_ci("H0_LKI")["95_upper"]:.2f}] (95% CL)')
print(f'  S8     = {get_mean("S8"):.4f} \u00b1 {get_std("S8"):.4f}  [{get_ci("S8")["95_lower"]:.4f}, {get_ci("S8")["95_upper"]:.4f}] (95% CL)')
print(f'  \u03c7\u00b2/dof = {total_from_col:.1f} / {n_dof} = {total_from_col/n_dof:.4f}')
print(f'\n  === Tensions ===')
print(f'  H0_LKI vs SH0ES: {results["tensions"]["H0_LKI_vs_SH0ES_sigma"]:.2f}\u03c3')
print(f'  H0_GKI vs SH0ES: {results["tensions"]["H0_GKI_vs_SH0ES_sigma"]:.2f}\u03c3')
print(f'  S8 vs DES:       {results["tensions"]["S8_vs_DES_sigma"]:.2f}\u03c3')


---


In [ ]:
# ============================================================
# §9. DOWNLOAD
# ============================================================
import zipfile, glob

_base = '/content'
zip_path = os.path.join(_base, 'NB05_D2_outputs.zip')
_count = 0

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for jpath in [os.path.join(_base, 'NB05_D2_results.json'), 'NB05_D2_results.json']:
        if os.path.exists(jpath):
            zf.write(jpath, 'NB05_D2_results.json')
            print(f'  Added: NB05_D2_results.json')
            _count += 1
            break
    for _fdir in [os.path.join(_base, 'figures'), 'figures']:
        if os.path.isdir(_fdir):
            for f in sorted(glob.glob(os.path.join(_fdir, 'fig_NB05_D2_*'))):
                arcname = os.path.join('figures', os.path.basename(f))
                zf.write(f, arcname)
                print(f'  Added: {arcname}')
                _count += 1
            break

assert _count > 0, 'FATAL: No files added to zip!'
print(f'\n[OK] NB05_D2_outputs.zip - {_count} files ({os.path.getsize(zip_path):,} bytes)')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f'Local env - file at {zip_path}')


---

## References

1. **Planck NPIPE** — CamSpec TTTEEE + lowl TT/EE + lensing
2. **DESI DR2** — BAO measurements (2024)
3. **Pantheon+** — Type Ia supernovae (Brout+ 2022)
4. **Cobaya** — Torrado & Lewis (2021)
5. **GetDist** — Lewis (2019)
6. **CLASS** — Blas, Lesgourgues & Tram (2011)
7. **Alvim (2025, 2026)** — Evaporating Universe, Paper I
